# TiniMind Training v3
LLM Bahasa Indonesia dari nol — pretrain + SFT

**Perubahan dari v2:**
- Training sekarang pakai `train.py` standalone (bisa dijalankan di mana saja)
- Notebook ini hanya setup + wrapper — logika training ada di `train.py`
- Format checkpoint **tetap sama** dengan v2, bisa resume dari checkpoint lama

**File yang dibutuhkan di Drive:**
```
TiniMind_Prototype/
├── model_v2.py
├── config.py
├── train.py          ← baru
├── quantize.py       ← baru
├── tokenizer/
│   └── indo_bpe_32k.model
└── data/
    └── mc4_indo/
        └── chunk_*.bin
```

In [11]:
# Cell 1 — Mount Drive & Setup
from google.colab import drive
drive.mount('/content/drive')

BASE     = '/content/drive/MyDrive/TiniMind_Prototype'
DATA_DIR = f'{BASE}/data/mc4_indo'
CKPT_DIR = f'{BASE}/output/checkpoints'
TOK_PATH = f'{BASE}/tokenizer/indo_bpe_32k.model'

import sys, os
sys.path.insert(0, BASE)

import torch
print(f'Device: {"cuda" if torch.cuda.is_available() else "cpu"}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB' if torch.cuda.is_available() else '')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cpu
GPU: none



In [12]:
# Cell 2 — Install dependencies
# sentencepiece untuk Indo BPE tokenizer
# datasets untuk streaming CulturaX kalau mau tambah data
%pip install -q sentencepiece datasets

In [13]:
# Cell 3 — Cek ketersediaan data & checkpoint
import glob, os

chunks = sorted(glob.glob(f'{DATA_DIR}/chunk_*.bin'))
total_tok = sum(os.path.getsize(f)//2 for f in chunks)
print(f'Chunks ditemukan : {len(chunks)}')
print(f'Total token      : {total_tok/1e9:.2f}B')

# Cek checkpoint terakhir untuk resume
ckpts = sorted(glob.glob(f'{CKPT_DIR}/step_*.pt'))
if ckpts:
    print(f'Checkpoint terakhir: {os.path.basename(ckpts[-1])}')
    RESUME = ckpts[-1]
else:
    print('Tidak ada checkpoint — training dari awal')
    RESUME = None

if not chunks:
    print('\n⚠️  Belum ada data! Jalankan Cell 4 (streaming) dulu.')

Chunks ditemukan : 301
Total token      : 3.00B
Tidak ada checkpoint — training dari awal


In [ ]:
# Cell 4 — Stream & tokenize CulturaX → chunk_*.bin
# SKIP kalau data/mc4_indo sudah punya chunk_*.bin yang cukup
#
# Format output: setiap chunk = array np.uint16 token ids, ~10M token per file
# Total target: 3B token = ~300 chunk (sudah tercapai di sesi sebelumnya)

import sentencepiece as spm
import numpy as np
from datasets import load_dataset

TARGET_TOKENS = 3_000_000_000   # 3B total
CHUNK_SIZE    = 10_000_000      # 10M per file

sp = spm.SentencePieceProcessor()
sp.Load(TOK_PATH)
print(f'Tokenizer vocab: {sp.GetPieceSize()}')

os.makedirs(DATA_DIR, exist_ok=True)
existing = sorted(glob.glob(f'{DATA_DIR}/chunk_*.bin'))
tokens_done = sum(os.path.getsize(f)//2 for f in existing)
print(f'Token sudah ada: {tokens_done/1e9:.2f}B / {TARGET_TOKENS/1e9:.1f}B')

if tokens_done >= TARGET_TOKENS:
    print('Target sudah tercapai, skip streaming.')
else:
    chunk_idx = len(existing)
    buf = []

    ds = load_dataset('uonlp/CulturaX', 'id', split='train', streaming=True, trust_remote_code=True)

    for doc in ds:
        toks = sp.Encode(doc['text'])
        buf.extend(toks)
        tokens_done += len(toks)

        while len(buf) >= CHUNK_SIZE:
            chunk = np.array(buf[:CHUNK_SIZE], dtype=np.uint16)
            path  = f'{DATA_DIR}/chunk_{chunk_idx:04d}.bin'
            chunk.tofile(path)
            print(f'Saved {os.path.basename(path)} | total: {tokens_done/1e9:.2f}B')
            buf = buf[CHUNK_SIZE:]
            chunk_idx += 1

        if tokens_done >= TARGET_TOKENS:
            print(f'Target {TARGET_TOKENS/1e9:.1f}B token tercapai!')
            break

    # Simpan sisa buffer
    if buf:
        path = f'{DATA_DIR}/chunk_{chunk_idx:04d}.bin'
        np.array(buf, dtype=np.uint16).tofile(path)
        print(f'Final chunk: {os.path.basename(path)}')

KeyboardInterrupt: 

In [14]:
RESUME_ARG = f'--resume {RESUME}' if RESUME else ''

os.makedirs(CKPT_DIR, exist_ok=True)

!python {BASE}/train.py \
    --config medium_130m \
    --data-dir {DATA_DIR} \
    --output-dir {CKPT_DIR} \
    --dtype fp16 \
    --max-steps 20000 \
    --lr 3e-4 \
    --batch-size 4 \
    --grad-accum 8 \
    --seq-len 1024 \
    --log-every 100 \
    --save-every 1000 \
    --eval-every 1000 \
    {RESUME_ARG}

Config: medium_130m | dtype: fp16 | device: cpu
max_steps=20000 | lr=0.0003 | batch=4 | grad_accum=8 | seq_len=1024
Chunks: 301 | Total: 3.00B token
Train: 299 chunks | Val: 2 chunks
Flash Attention aktif (F.scaled_dot_product_attention) | vocab=100352
Params: 100.7M
/content/drive/MyDrive/TiniMind_Prototype/train.py:214: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  scaler = GradScaler(enabled=(args.dtype == "fp16"))
^C


In [ ]:
# Cell 6 — Supervised Fine-Tuning (SFT)
#
# Format data SFT: JSONL, setiap baris {"turns": [["user", "..."], ["assistant", "..."]]}
# (format yang sudah kamu buat di sesi sebelumnya)
#
# Tokenize SFT data dulu jadi chunk_*.bin di folder terpisah,
# lalu jalankan train.py dengan --data-dir ke folder SFT.

SFT_DATA   = f'{BASE}/data/sft'
SFT_CKPT   = f'{BASE}/output/sft'
SFT_JSONL  = f'{BASE}/data/sft.jsonl'   # file SFT yang sudah dibuat

import sentencepiece as spm, json
import numpy as np, os

os.makedirs(SFT_DATA, exist_ok=True)
os.makedirs(SFT_CKPT, exist_ok=True)

sp = spm.SentencePieceProcessor()
sp.Load(TOK_PATH)

# Tokenize semua conversation jadi satu bin
# Format: <penggunna> teks </penggunna> <asisten> teks </asisten>
# (sesuai special token di train_tokenizer_indo.py)
all_tokens = []
with open(SFT_JSONL) as f:
    for line in f:
        conv = json.loads(line)
        text = ''
        for role, content in conv['turns']:
            if role == 'user':
                text += f'<penggunna>{content}</penggunna>'
            else:
                text += f'<asisten>{content}</asisten>'
        all_tokens.extend(sp.Encode(text))

arr = np.array(all_tokens, dtype=np.uint16)
arr.tofile(f'{SFT_DATA}/chunk_0000.bin')
print(f'SFT tokens: {len(all_tokens):,} | saved ke chunk_0000.bin')

# Ambil checkpoint pretrain terbaik untuk fine-tune
pretrain_ckpts = sorted(glob.glob(f'{CKPT_DIR}/step_*.pt'))
best_pretrain   = pretrain_ckpts[-1] if pretrain_ckpts else None
print(f'Mulai SFT dari: {os.path.basename(best_pretrain) if best_pretrain else "TIDAK ADA"}')

In [ ]:
# Cell 6b — Jalankan SFT
SFT_RESUME_ARG = f'--resume {best_pretrain}' if best_pretrain else ''

!python {BASE}/train.py \
    --config medium_130m \
    --data-dir {SFT_DATA} \
    --output-dir {SFT_CKPT} \
    --dtype fp16 \
    --max-steps 500 \
    --lr 1e-5 \
    --batch-size 2 \
    --grad-accum 4 \
    --seq-len 1024 \
    --log-every 50 \
    --save-every 100 \
    --eval-every 100 \
    --val-chunks 0 \
    {SFT_RESUME_ARG}

In [ ]:
# Cell 7 — Quick inference test
import torch, sys, glob, os
sys.path.insert(0, BASE)
from model_v2 import TiniMind
from config import ModelConfig
import sentencepiece as spm

# Load checkpoint terbaik
sft_ckpts = sorted(glob.glob(f'{SFT_CKPT}/step_*.pt'))
ckpt_path = sft_ckpts[-1] if sft_ckpts else sorted(glob.glob(f'{CKPT_DIR}/step_*.pt'))[-1]
print(f'Load: {os.path.basename(ckpt_path)}')

ckpt = torch.load(ckpt_path, map_location='cpu')
cfg  = ckpt.get('config', ModelConfig(num_layers=24, hidden_size=1024,
                                       num_heads=16, num_kv_heads=4,
                                       vocab_size=32000, max_seq_len=2048))

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model  = TiniMind(cfg).to(device)
model.load_state_dict(ckpt['model'])
model.eval()
print(f'Model loaded: {model.num_params()/1e6:.1f}M params')

sp = spm.SentencePieceProcessor()
sp.Load(TOK_PATH)

@torch.no_grad()
def generate(prompt: str, max_new_tokens: int = 200, temperature: float = 0.8, top_k: int = 50) -> str:
    ids = torch.tensor([sp.Encode(prompt)], dtype=torch.long).to(device)
    past_kvs = None

    for _ in range(max_new_tokens):
        logits, _, past_kvs = model(ids if past_kvs is None else ids[:, -1:],
                                     use_kv_cache=True, past_kvs=past_kvs,
                                     offset=0 if past_kvs is None else ids.shape[1] - 1)
        logits = logits[:, -1, :] / temperature
        if top_k:
            v, _ = torch.topk(logits, top_k)
            logits[logits < v[:, -1:]] = -float('inf')
        probs  = torch.softmax(logits, dim=-1)
        nxt    = torch.multinomial(probs, 1)
        ids    = torch.cat([ids, nxt], dim=1)

    return sp.Decode(ids[0].tolist())

# Test
prompt = '<penggunna>Jelaskan apa itu kecerdasan buatan?</penggunna><asisten>'
print(generate(prompt))

In [ ]:
# Cell 8 — Quantize INT8 untuk inferensi lebih cepat
# Mode dynamic: CPU only, tidak butuh bitsandbytes
# Mode bnb8bit: GPU, butuh pip install bitsandbytes

QUANTIZED_OUT = f'{BASE}/output/tinimind_int8.pt'

!python {BASE}/quantize.py \
    --checkpoint {ckpt_path} \
    --output {QUANTIZED_OUT} \
    --mode dynamic